# Chapter 16: Hallucination Detection via Geodesic Distance

**Learning objectives:**
- Understand hallucination as geometric distance on the GMS manifold
- Use `HallucinationOracle` to classify claims as fabrication/distortion/embellishment/grounded
- Generate adversarial hallucination probes via `HallucinationGenerator`
- Measure aggregate hallucination rates across a full DOE benchmark

---
## Preamble

Part IV built the black-box testing stack: adapters, gateway, policy, environment and oracles. Part V turns to **behavioral and quality testing** — evaluating *what* the agent says, not just *how* it behaves.

The first behavioral concern is **hallucination**: the agent generates claims that are geometrically far from any known fact on the GMS manifold. GMSH measures this distance via `score_triple()` and classifies severity by geodesic thresholds:

| Geodesic | Severity | Meaning |
|----------|----------|--------|
| > 1.5 | fabrication | Entirely invented content |
| 0.7 – 1.5 | distortion | Significantly altered facts |
| 0.5 – 0.7 | embellishment | Minor additions beyond source |
| < 0.5 | grounded | Consistent with known facts |

In [ ]:
# === Preamble (all notebooks) ===
import warnings
warnings.filterwarnings("ignore")

import torch
import numpy as np
from collections import Counter

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
from importlib.resources import files
# Load the Chapter 4 store
from knowlytix.harness.doe import DOEGMSBenchmark, DOEHarnessConfig

config = DOEHarnessConfig(
    markdown_path=str(files("knowlytix.harness.doe").joinpath("tests/model_risk_assessment.md")),
    stores_dir="monograph/lab_stores",
    store_name="ch04_ground_truth",
    n_runs=4,
    ingest_epochs=200,
    device=str(DEVICE),
)
bench = DOEGMSBenchmark(config)
store = bench.ingest()
print(f"Store: {store.stats()['entities']} entities, "
      f"{store.stats()['triples']} triples, "
      f"{store.stats()['enm_entries']} ENM")

---

## Hallucination as Geometric Distance

A **hallucination** is a claim whose `score_triple()` geodesic distance is large — the claim is far from any learned fact on the manifold $S^{m-1}$. The `HallucinationOracle` scores each claim as a (head, relation, tail) triple and classifies it into severity tiers.

Key data structures:

| Class | Fields | Purpose |
|---|---|---|
| `HallucinationAssessment` | claim_text, head, relation, tail, geodesic_distance, tension_vs_nearest, severity | Per-claim result |
| `HallucinationReport` | n_claims, n_hallucinated, hallucination_rate, mean_geodesic, max_geodesic, severity_counts, assessments | Aggregate report |
| `HallucinationOracle` | store, geodesic_threshold | Oracle that scores claims |
| `HallucinationGenerator` | — | Generates adversarial probes |

In [ ]:
# Score individual claims against the manifold
from knowlytix.harness.doe.hallucination import (
    HallucinationOracle, HallucinationGenerator,
)

oracle = HallucinationOracle(store=store, geodesic_threshold=1.0)

# A known triple from the store — should have low geodesic
a_known = oracle.assess_claim(
    head="MDL-001", relation="in_section", tail="performance_metrics"
)
print("=== Known Triple ===")
print(f"  Claim:    {a_known.claim_text}")
print(f"  Geodesic: {a_known.geodesic_distance:.4f}")
print(f"  Severity: {a_known.severity}")

# A fabricated triple — entity exists but tail is invented
a_corrupt = oracle.assess_claim(
    head="MDL-001", relation="has_algorithm", tail="QuantumBoosting"
)
print("\n=== Fabricated Triple ===")
print(f"  Claim:    {a_corrupt.claim_text}")
print(f"  Geodesic: {a_corrupt.geodesic_distance:.4f}")
print(f"  Severity: {a_corrupt.severity}")

---

## Severity Tiers Across Known and Fabricated Claims

To understand the geodesic landscape, we score a sample of known triples from the store and compare against corrupted and fabricated claims.

In [ ]:
# Score known triples from the store
import random
random.seed(42)

triples = list(store.doc_graph.triples)
random.shuffle(triples)

known_assessments = []
for h, r, t in triples[:100]:
    a = oracle.assess_claim(h, r, t)
    known_assessments.append(a)

known_tiers = Counter(a.severity for a in known_assessments)
known_geos = [a.geodesic_distance for a in known_assessments]

print("=== Known Triples (100 sampled) ===")
print(f"  Mean geodesic: {np.mean(known_geos):.4f}")
print(f"  Std geodesic:  {np.std(known_geos):.4f}")
print(f"  Min geodesic:  {np.min(known_geos):.4f}")
print(f"  Max geodesic:  {np.max(known_geos):.4f}")
print(f"  Severity counts: {dict(sorted(known_tiers.items()))}")

In [ ]:
# Score fabricated claims — entities not in the store
fabricated_claims = [
    ("MDL-001", "has_algorithm", "QuantumBoosting"),
    ("CreditScore-XGB", "has_detail", "Alan_Turing"),
    ("NonexistentModel", "has_risk_tier", "Ultra-Critical"),
    ("FraudNet-LSTM", "invented_by", "deep_learning_lab"),
    ("MDL-099", "has_accuracy", "0.9999"),
]

report_fab = oracle.assess_triples(fabricated_claims)

print("=== Fabricated Claims ===")
print(f"  Claims:            {report_fab.n_claims}")
print(f"  Hallucinated:      {report_fab.n_hallucinated}")
print(f"  Hallucination rate: {report_fab.hallucination_rate:.0%}")
print(f"  Mean geodesic:     {report_fab.mean_geodesic:.4f}")
print(f"  Max geodesic:      {report_fab.max_geodesic:.4f}")
print(f"  Severity counts:   {report_fab.severity_counts}")

print(f"\n{'Claim':<55s} {'Geodesic':>8s} {'Severity'}")
print("-" * 80)
for a in report_fab.assessments:
    print(f"{a.claim_text:<55s} {a.geodesic_distance:>8.4f} {a.severity}")

In [ ]:
# Mixed batch: grounded + corrupted + fabricated
grounded_claims = [
    ("MDL-001", "in_section", "performance_metrics"),
    ("MDL-001", "in_section", "model_inventory_summary"),
]

corrupted_claims = [
    ("MDL-002", "in_section", "3.0"),           # wrong section
    ("MDL-004", "has_delta_auc", "High"),        # wrong type
    ("Organization", "has_detail", "47"),         # swapped detail
]

fully_fabricated = [
    ("MDL-001", "has_algorithm", "QuantumBoosting"),
    ("NonexistentModel", "has_risk_tier", "Ultra-Critical"),
]

all_claims = grounded_claims + corrupted_claims + fully_fabricated
labels = (["grounded"] * len(grounded_claims) +
          ["corrupted"] * len(corrupted_claims) +
          ["fabricated"] * len(fully_fabricated))

report_mixed = oracle.assess_triples(all_claims)

print("=== Mixed Batch (7 claims) ===")
print(f"  Hallucination rate: {report_mixed.hallucination_rate:.0%}")
print(f"  Severity counts:   {report_mixed.severity_counts}")

print(f"\n{'Source':<12s} {'Claim':<50s} {'Geo':>7s} {'Severity'}")
print("-" * 85)
for label, a in zip(labels, report_mixed.assessments):
    print(f"{label:<12s} {a.claim_text:<50s} {a.geodesic_distance:>7.4f} {a.severity}")

---

## Lab 16.1 — Detect Hallucinations Geometrically

In this lab we assess hallucination across multiple geodesic thresholds to understand how threshold choice affects detection sensitivity.

In [ ]:
# Lab 16.1a — Threshold sensitivity analysis
# Score all 100 known triples + 5 fabricated claims at different thresholds
all_test_claims = [(h, r, t) for h, r, t in triples[:100]]
all_test_claims += fabricated_claims
claim_sources = ["known"] * 100 + ["fabricated"] * 5

# Assess with default oracle (threshold=1.0)
all_assessments = [oracle.assess_claim(h, r, t) for h, r, t in all_test_claims]
all_geos = [a.geodesic_distance for a in all_assessments]

thresholds = [0.5, 0.7, 1.0, 1.5, 2.0]
print(f"{'Threshold':<12s} {'Hallucinated':>12s} {'Rate':>8s} {'Known flagged':>14s} {'Fab flagged':>12s}")
print("-" * 65)
for th in thresholds:
    n_halluc = sum(1 for g in all_geos if g > th)
    known_flagged = sum(1 for g, s in zip(all_geos[:100], claim_sources[:100]) if g > th)
    fab_flagged = sum(1 for g, s in zip(all_geos[100:], claim_sources[100:]) if g > th)
    print(f"{th:<12.1f} {n_halluc:>12d} {n_halluc/len(all_geos):>8.1%} "
          f"{known_flagged:>14d} {fab_flagged:>12d}")

In [ ]:
# Lab 16.1b — Severity distribution for known vs fabricated
known_sevs = Counter(a.severity for a in all_assessments[:100])
fab_sevs = Counter(a.severity for a in all_assessments[100:])

tiers = ["grounded", "embellishment", "distortion", "fabrication"]
print(f"{'Severity':<15s} {'Known (100)':>12s} {'Fabricated (5)':>15s}")
print("-" * 45)
for t in tiers:
    print(f"{t:<15s} {known_sevs.get(t, 0):>12d} {fab_sevs.get(t, 0):>15d}")

---

## The HallucinationGenerator

The `HallucinationGenerator` creates questions designed to elicit hallucination via three strategies:

1. **Boundary entities** — entities with few triples (weakly anchored on $S^{m-1}$)
2. **Near-miss entities** — plausible but nonexistent entity names
3. **Open-ended elaboration** — "explain in detail" prompts for well-known entities

In [ ]:
# Generate hallucination probes
gen = HallucinationGenerator()
probes = gen.generate(store)

strategies = Counter(p.metadata["strategy"] for p in probes)
print(f"Generated {len(probes)} hallucination probes")
print(f"By strategy: {dict(sorted(strategies.items()))}")

print(f"\n{'Strategy':<20s} {'QID':<30s} {'Entity/Target'}")
print("-" * 80)
for p in probes[:5]:
    entity = p.metadata.get("entity", p.metadata.get("fake_entity", "?"))
    print(f"{p.metadata['strategy']:<20s} {p.qid:<30s} {entity}")
print("...")
for p in probes[10:13]:
    entity = p.metadata.get("entity", p.metadata.get("fake_entity", "?"))
    print(f"{p.metadata['strategy']:<20s} {p.qid:<30s} {entity}")
print("...")
for p in probes[15:18]:
    entity = p.metadata.get("entity", p.metadata.get("fake_entity", "?"))
    print(f"{p.metadata['strategy']:<20s} {p.qid:<30s} {entity}")

---

## Lab 16.2 — Hallucination Probes on Mock Agent

In this lab we run the hallucination probes through the `FullMockHarness`, extract claims from the agent output and assess them with the `HallucinationOracle`.

In [ ]:
from importlib.resources import files
# Lab 16.2a — Run probes through the mock agent
from knowlytix.harness.doe.adapters import AgentAdapter
from knowlytix.harness.fixtures import FullMockHarness

adapter = AgentAdapter(
    harness=FullMockHarness(recovery_mode="fallback"),
)
context = open(str(files("knowlytix.harness.doe").joinpath("tests/model_risk_assessment.md"))).read()

# Collect traces for all probes
probe_traces = []
for p in probes:
    trace = adapter.evaluate(question=p.natural_language, context=context)
    probe_traces.append((p, trace))

print(f"Collected {len(probe_traces)} probe traces")
print("\nSample outputs:")
for p, t in probe_traces[:3]:
    print(f"  [{p.metadata['strategy']}] {p.qid}: {t.raw_output!r}")

In [ ]:
# Lab 16.2b — Assess agent outputs as hallucination claims
# The mock agent returns a numeric score — we treat (entity, "has_answer", output)
# as the claim triple, where 'entity' comes from the probe metadata.

probe_reports = []
for p, t in probe_traces:
    entity = p.metadata.get("entity", p.metadata.get("fake_entity", "unknown"))
    claim = (entity, "has_answer", t.raw_output.strip())
    report = oracle.assess_triples([claim])
    probe_reports.append((p, report))

# Aggregate by strategy
by_strategy = {}
for p, report in probe_reports:
    strat = p.metadata["strategy"]
    by_strategy.setdefault(strat, []).append(report)

print(f"{'Strategy':<20s} {'N':>4s} {'Halluc Rate':>12s} {'Mean Geo':>10s} {'Severity Distribution'}")
print("-" * 85)
for strat in sorted(by_strategy):
    reports = by_strategy[strat]
    n = len(reports)
    halluc_rate = np.mean([r.hallucination_rate for r in reports])
    mean_geo = np.mean([r.mean_geodesic for r in reports])
    sevs = Counter()
    for r in reports:
        for a in r.assessments:
            sevs[a.severity] += 1
    print(f"{strat:<20s} {n:>4d} {halluc_rate:>12.0%} {mean_geo:>10.4f} {dict(sorted(sevs.items()))}")

In [ ]:
# Lab 16.2c — Per-probe detail
print(f"{'QID':<30s} {'Strategy':<18s} {'Geo':>7s} {'Severity':<14s} {'Entity'}")
print("-" * 95)
for p, report in probe_reports:
    a = report.assessments[0]
    entity = p.metadata.get("entity", p.metadata.get("fake_entity", "?"))
    print(f"{p.qid:<30s} {p.metadata['strategy']:<18s} "
          f"{a.geodesic_distance:>7.4f} {a.severity:<14s} {entity}")

---

## End-to-End: 640 Questions with Hallucination Detection

We run the full DOE benchmark and assess every trace for hallucination. Each agent response is treated as a claim triple `(question_entity, "has_answer", output)` and scored against the manifold.

In [ ]:
# E2E: 640 questions with hallucination assessment
doe_questions = bench.generate_questions()

e2e_reports = []
for dq in doe_questions:
    trace = adapter.evaluate(
        question=dq.question.natural_language,
        context=context,
    )
    # Extract entity from question metadata or ground truth
    head = dq.question.ground_truth if dq.question.ground_truth else "agent"
    claim = (str(head), "has_answer", trace.raw_output.strip())
    report = oracle.assess_triples([claim])
    e2e_reports.append(report)

print(f"Assessed {len(e2e_reports)} agent responses")

In [ ]:
# Aggregate hallucination analysis
e2e_geos = [r.mean_geodesic for r in e2e_reports]
e2e_halluc = sum(1 for r in e2e_reports if r.hallucination_rate > 0)
e2e_sevs = Counter()
for r in e2e_reports:
    for a in r.assessments:
        e2e_sevs[a.severity] += 1

print(f"=== Hallucination Summary ({len(e2e_reports)} questions) ===")
print(f"  Hallucinated responses: {e2e_halluc} ({100*e2e_halluc/len(e2e_reports):.1f}%)")
print(f"  Grounded responses:     {len(e2e_reports)-e2e_halluc} ({100*(len(e2e_reports)-e2e_halluc)/len(e2e_reports):.1f}%)")

print("\n=== Geodesic Distance Distribution ===")
print(f"  Mean   : {np.mean(e2e_geos):.4f}")
print(f"  Std    : {np.std(e2e_geos):.4f}")
print(f"  Median : {np.median(e2e_geos):.4f}")
print(f"  Min    : {np.min(e2e_geos):.4f}")
print(f"  Max    : {np.max(e2e_geos):.4f}")

print("\n=== Severity Distribution ===")
tiers = ["grounded", "embellishment", "distortion", "fabrication"]
for t in tiers:
    count = e2e_sevs.get(t, 0)
    print(f"  {t:<15s}: {count:>4d} ({100*count/len(e2e_reports):.1f}%)")

In [ ]:
# Threshold sweep on E2E results
print(f"{'Threshold':<12s} {'Halluc Count':>12s} {'Rate':>8s}")
print("-" * 35)
for th in [0.5, 0.7, 1.0, 1.5, 2.0]:
    n = sum(1 for g in e2e_geos if g > th)
    print(f"{th:<12.1f} {n:>12d} {n/len(e2e_geos):>8.1%}")

---

## Exercises

**Exercise 16.1** — Create a `HallucinationOracle` with `geodesic_threshold=0.5` (strict) and re-assess the 100 known triples from Section 16.2. How many known facts get flagged as hallucinations? What does this tell you about threshold selection?

**Exercise 16.2** — Use the `HallucinationGenerator` to generate probes. For each probe, construct three artificial agent responses: one grounded (repeats known facts), one embellished (adds plausible detail) and one fabricated (invents entities). Assess all three with the oracle and verify the severity tiers separate correctly.

**Exercise 16.3** — Run the 640-question benchmark with two different `geodesic_threshold` values (0.7 and 1.5). Compare the hallucination rates and severity distributions. At which threshold does the false-positive rate on known entities become acceptable for a production release gate?

---

## Summary

This chapter introduced **hallucination detection via geodesic distance** — the first behavioral testing capability in Part V:

1. **Hallucination as geometry**: a claim whose `score_triple()` geodesic is large lies far from the learned manifold, indicating fabrication.

2. **Four severity tiers** classify claims by geodesic distance: grounded (< 0.5), embellishment (0.5–0.7), distortion (0.7–1.5) and fabrication (> 1.5).

3. **HallucinationOracle** provides `assess_claim()` for single triples and `assess_triples()` for batch assessment, returning structured `HallucinationReport` with per-claim `HallucinationAssessment` objects.

4. **HallucinationGenerator** creates adversarial probes via three strategies: boundary entities (weakly anchored), near-miss entities (plausible fakes) and open-ended elaboration (cascading hallucination triggers).

5. **Threshold selection** is critical: a strict threshold (0.5) catches embellishments but flags many known facts; a lenient threshold (1.5) only catches fabrications. The choice depends on the risk tier and regulatory requirements.

In the next chapter we turn to **confidence calibration** — measuring whether the agent's claimed confidence matches its actual accuracy using phase encoding on $S^1$.